In [39]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Load Dataset

In [40]:
df = pd.read_csv("ecommerce_fraud_dataset.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (1000, 14)


,total_orders,returned_orders,return_rate,cancelled_orders,avg_order_value,seller_total_sales,seller_returns,seller_return_rate,avg_rating,negative_review_percent,reviews_same_seller,repeated_review_similarity,fraud_label,fraud_type
0,107,51,0.477,28,77320,1658,121,0.073,1.62,5.81,24,0.334,0,NONE
1,108,23,0.213,2,67721,1352,769,0.569,3.89,93.86,2,0.182,0,NONE
2,25,0,0.000,3,29193,272,88,0.324,2.16,61.19,42,0.047,0,NONE
3,64,15,0.234,14,66225,977,562,0.575,2.53,98.32,3,0.860,1,REPEATED_REVIEW
4,11,4,0.364,0,11127,620,315,0.508,3.25,38.54,26,0.098,0,NONE


Fraud Type Distribution

In [41]:
print(df["fraud_type"].value_counts())

fraud_type
NONE                 589
SELLER_SUSPICIOUS    221
REPEATED_REVIEW      110
RETURN_ABUSE          80
Name: count, dtype: int64


Features and Target

In [42]:
X = df[
    [
        "total_orders",
        "returned_orders",
        "return_rate",
        "cancelled_orders",
        "avg_order_value",
        "seller_total_sales",
        "seller_returns",
        "seller_return_rate",
        "avg_rating",
        "negative_review_percent",
        "reviews_same_seller",
        "repeated_review_similarity"
    ]
]

y = df["fraud_type"]

Encode Target Labels

In [43]:
label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

Classes:
['NONE' 'REPEATED_REVIEW' 'RETURN_ABUSE' 'SELLER_SUSPICIOUS']


Split Dataset

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 800
Testing Samples: 200


Train Random Forest

In [45]:
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced"]
}

rf = RandomForestClassifier(
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="f1_weighted",
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Score:")
print(random_search.best_score_)

model = random_search.best_estimator_

print("\nModel Training Complete")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 30, 'class_weight': 'balanced'}

Best Score:
0.8772844090866811

Model Training Complete


Evaluate Model

In [46]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 88.5 %


Detailed Report

In [47]:
print(
    classification_report(
        y_test,
        predictions,
        target_names=label_encoder.classes_
    )
)

                   precision    recall  f1-score   support

             NONE       0.89      0.94      0.91       118
  REPEATED_REVIEW       0.95      0.86      0.90        22
     RETURN_ABUSE       0.85      0.69      0.76        16
SELLER_SUSPICIOUS       0.86      0.82      0.84        44

         accuracy                           0.89       200
        macro avg       0.89      0.83      0.85       200
     weighted avg       0.88      0.89      0.88       200



Confusion Matrix

In [48]:
cm = confusion_matrix(y_test, predictions)

print(cm)

[[111   1   1   5]
 [  1  19   1   1]
 [  5   0  11   0]
 [  8   0   0  36]]


Save Model

In [49]:
joblib.dump(model, "fraud_model.pkl")
joblib.dump(label_encoder, "fraud_label_encoder.pkl")

print("Model Saved Successfully")

Model Saved Successfully
